# Humanoid training: local and Colab share the same experiment

Select a GPU runtime. Upload the exported bundle folder to Drive and set BUNDLE_DIR below. The bundle contains the current working source, including uncommitted changes. Use `--bundle-data` when exporting to include the required motions/caches. Training uses the VM disk; completed checkpoints are copied to Drive periodically.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
BUNDLE_DIR = Path('/content/drive/MyDrive/humanoid_rl/colab/REPLACE_WITH_BUNDLE_FOLDER')


In [ ]:
import hashlib, json, subprocess, sys, zipfile
bundle = json.loads((BUNDLE_DIR / 'bundle.json').read_text())
REPO = Path('/content') / ('humanoid_rl_' + bundle['run_id'])
REPO.mkdir(parents=True, exist_ok=True)
def extract_verified(name, destination):
    path = BUNDLE_DIR / name
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1048576), b''):
            digest.update(block)
    if digest.hexdigest() != bundle['archives'][name]:
        raise ValueError('Archive checksum differs: ' + name)
    with zipfile.ZipFile(path) as archive:
        for info in archive.infolist():
            if not (destination / info.filename).resolve().is_relative_to(destination.resolve()):
                raise ValueError('Archive path escapes destination')
        archive.extractall(destination)
extract_verified('source.zip', REPO)
if 'data.zip' in bundle['archives']:
    extract_verified('data.zip', REPO)
if 'resume.zip' in bundle['archives']:
    extract_verified('resume.zip', REPO / '_resume')
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
subprocess.run([sys.executable, str(REPO / 'humanoid.py'), 'setup',
                '--method', bundle['method'], '--task', bundle['task']], cwd=REPO, check=True)


## Run or resume

The experiment includes CLI overrides from export. The exact resolved configuration, environment versions and source fingerprints are saved in the run folder. If data.zip was omitted, copy the inputs listed in inputs.json under REPO first. For a later session, set RESUME to a checkpoint copied back from Drive. Choose a new RUN_ID for each invocation; the saved weights continue into a new run folder.


In [ ]:
RUN_ID = bundle['run_id']
RESUME = str(REPO / '_resume' / bundle['resume']) if bundle.get('resume') else None
command = [sys.executable, str(REPO / 'humanoid.py'), 'train',
           '--experiment', str(BUNDLE_DIR / 'experiment.json'),
           '--runtime', str(BUNDLE_DIR / 'runtime.json'), '--run-id', RUN_ID]
if RESUME:
    command += ['--resume', RESUME]
for override in bundle.get('overrides', []):
    command += ['--set', override]
subprocess.run(command, cwd=REPO, check=True)
